# Reference-free spatial deconvolution

Learn component profiles and spatial molecular weights from your raw counts.
For exposure $s_i=\sum_gY_{ig}$, the model reconstructs $Y_{ig}$ as $s_i(PB)_{ig}$.
Rows of $P$ and $B$ sum to one. Three molecular initializations are fitted independently,
aligned and averaged before joint refinement with coordinate-based spatial TV.

Required input: `counts.h5ad`, with counts and `obsm["spatial"]` coordinates.
Optional evaluation: a CSV of known fractions, with spot IDs in its first column and one column per type.
Component weights describe molecular contributions; they are not automatically cell-count fractions.

## Data links

- Cerebellum: [SCP354 / Slide-seq](https://singlecell.broadinstitute.org/single_cell/study/SCP354/slide-seq-study).
- Cell counts and labels: [RETROFIT benchmark files](https://github.com/qunhualilab/retrofit-scripts).

## Example figures

Archived cerebellum example: known and inferred compositions on an artificial pseudo-spot
display grid, followed by correlations after matching components to cell types.

![Known and inferred cerebellum pseudo-spot compositions](../docs/source/_static/figures/cerebellum_pies.png)

![Matched cerebellum cell-type and component correlations](../docs/source/_static/figures/cerebellum_correlation.png)


## 1. Configure your input

Install the notebook dependencies, then edit `DATA_DIR`, the input filenames and `K` below.
Supply your own `.h5ad` files with unique spot and feature IDs. RNA fitting uses raw counts;
set `COUNTS_LAYER="counts"` when they are stored in a layer. Protein inputs should contain
your prepared nonnegative assay measurements. Each fitted spot needs positive total signal.

Running the fitting cells trains a new model. `DEVICE` accepts `"auto"`, `"cpu"` or `"cuda"`.
The study's reference-free and joint analyses used GPUs. Inspect the returned diagnostics
to determine whether your fit converged. Large inputs require enough memory for dense matrices.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from IPython.display import display
from tutorials.common import correlation, align, spatial, pies, legend, save_figure, save_table
from spDecon.reproducibility import set_seed

DATA_DIR = Path("/path/to/your/data")
COUNTS_LAYER = None
DEVICE = "auto"
VERBOSE = True
set_seed(1, deterministic=True)

K = 11  # Choose the component number for your data.
TRUTH_CSV = None  # Optional: DATA_DIR / "fractions.csv"


## 2. Read your counts

Coordinates must describe the spatial geometry used for smoothing. An artificial display grid
should not be used as an anatomical neighborhood graph. `section_id` keeps graphs within sections.
For bulk mixtures without spatial coordinates, use the bulk workflow in the API guide.


In [ ]:
adata = ad.read_h5ad(DATA_DIR / "counts.h5ad")
y = np.array(adata.to_df(layer=COUNTS_LAYER), dtype=float, order="F")
xy = np.asarray(adata.obsm["spatial"])
sections = adata.obs["section_id"].to_numpy() if "section_id" in adata.obs else np.repeat("section_1", len(y))
print(adata)


## 3. Initialize, calibrate and fit


In [ ]:
from spDecon.config import SolverConfig
from spDecon.consensus import build_consensus
from spDecon.spatial import spatial_graph, calibrate, refine
initial = build_consensus(y, K, random_state=1,
    solver_config=SolverConfig(device=DEVICE, random_state=1, verbose=VERBOSE))
i, j, w, graph_info = spatial_graph(xy, sections)
calibration = calibrate(y, initial.consensus_proportions, initial.consensus_profiles, i, j, w, device=DEVICE)
result = refine(y, initial.consensus_proportions, initial.consensus_profiles,
    i, j, w, calibration["spatial_weight"], 1, device=DEVICE, verbose=VERBOSE)
p, b = result.proportions, result.profiles
components = [f"C{k+1}" for k in range(K)]
display(result.diagnostics)


## 4. Inspect and save component maps


In [ ]:
fig, axes = plt.subplots(int(np.ceil(p.shape[1]/4)), 4,
                         figsize=(12, 3*int(np.ceil(p.shape[1]/4))), squeeze=False)
for k, ax in enumerate(axes.ravel()):
    if k < p.shape[1]:
        art = spatial(ax, xy, p[:,k], components[k], size=10)
    else:
        ax.axis("off")
fig.colorbar(art, ax=axes, label="Molecular contribution", shrink=.6)
save_figure(fig, "reference_free_components")


In [ ]:
adata.obsm["spDecon"] = p
adata.varm["spDecon_profiles"] = b.T
adata.uns["spDecon_components"] = components
out = ROOT / "outputs/reference_free"
out.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(out / "deconvolved.h5ad")


## 5. Optional evaluation against known fractions

Leave `TRUTH_CSV=None` when no ground truth is available. For a complete one-to-one comparison,
the CSV needs the same spots and `K` types. Matching is performed only after fitting.
These are descriptive recovery scores on the supplied evaluation data.


In [ ]:
if TRUTH_CSV is not None:
    truth = pd.read_csv(TRUTH_CSV, index_col=0).loc[adata.obs_names]
    if truth.shape != p.shape:
        raise ValueError("Evaluation needs one truth column per fitted component")
    matched, order = align(p, truth.to_numpy())
    scores = pd.DataFrame({"type": truth.columns,
        "PCC": np.diag(correlation(matched, truth.to_numpy()))})
    display(save_table(scores, "reference_free_pcc"))
    print("RMSE:", np.sqrt(np.mean((matched-truth.to_numpy())**2)))
else:
    print("No fraction labels supplied; component weights are available above.")
